<a href="https://colab.research.google.com/github/Hyounseo/Opensource-FISH/blob/main/Url_%ED%83%90%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import re
import pandas as pd
import numpy as np

# [2단계-A] 편집 거리(Levenshtein Distance) 직접 구현
# [2단계-A]편집 거리(Levenshtein Distance) 직접 구현
def get_levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return get_levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]


# [2단계-B]539개 정상 도메인 DB 파일 로드
# ⚠️ 파일명이 'correct_domain.csv'이고 'FISH_dataset' 폴더 안에 있는지 꼭 확인하세요!
csv_path = '/content/drive/MyDrive/FISH_dataset/correct_domain.csv'

print("--- 📂 정식 도메인 화이트리스트 데이터셋 로드 시작 ---")
if os.path.exists(csv_path):
    # 첫 줄 naver.com 누락 방지를 위한 header=None 처리 패치 적용
    df_domains = pd.read_csv(csv_path, header=None, names=['official_domain'])
    official_domains_db = df_domains['official_domain'].dropna().tolist()
    print(f"✅ 데이터셋 로드 완료! 총 {len(official_domains_db)}개의 정식 도메인이 준비되었습니다.")
else:
    print(f"❌ [에러] CSV 파일을 찾을 수 없습니다: {csv_path}")
    print("💡 경로가 다를 경우 아래 백업 데이터셋으로 자동 대체하여 연산합니다.")
    official_domains_db = ["naver.com", "daum.net", "google.com", "kakaobank.com", "toss.im", "coupang.com"]

--- 📂 정식 도메인 화이트리스트 데이터셋 로드 시작 ---
✅ 데이터셋 로드 완료! 총 539개의 정식 도메인이 준비되었습니다.


In [4]:
# =====================================================================
# [최종 완결] 도메인 매칭 버그 완벽 수정 버전
# =====================================================================
!pip install -q easyocr scikit-learn gradio

import os, re, easyocr, pandas as pd
from google.colab import files
from urllib.parse import urlparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import gradio as gr

# ---------------- 구글 드라이브 마운트 추가 ----------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ---------------- CSV 로드 ----------------
correct_domain_path = '/content/drive/MyDrive/FISH_dataset/correct_domain.csv'
df_domains = pd.read_csv(correct_domain_path, header=None, names=['official_domain'])
official_domains_db = df_domains['official_domain'].dropna().str.strip().str.lower().tolist()
official_domains_db = [d for d in official_domains_db if d != 'correct_domain']  # CSV 헤더 행이 데이터로 섞여 들어가는 것 방지
official_domains_db = list(set(official_domains_db))

# 화이트리스트 전처리는 호출마다 반복하지 않고 미리 한 번만 만들어둠 (성능)
official_domains_clean = [str(d).strip().lower() for d in official_domains_db if str(d).strip()]

extracted_path = '/content/drive/MyDrive/FISH_dataset/extracted_data.csv'
df = pd.read_csv(extracted_path)

# ---------------- 도메인 정제 함수 ----------------
def normalize_url_text(text):
    text = str(text).lower()
    text = re.sub(r'\|\|+', '', text)          # || 제거
    text = re.sub(r'\.{2,}', '.', text)        # 연속된 점 줄이기
    text = re.sub(r'[^a-z0-9://.\-]', '', text) # 불필요한 특수문자 제거
    #  OCR이 ://를 sll, sl, ll 등으로 잘못 읽는 경우 보정
    text = re.sub(r'https?sll', 'https://', text)
    text = re.sub(r'https?sl(?!l)', 'https://', text)
    text = re.sub(r'http(?!s)sll', 'http://', text)

    text = re.sub(r'\bww+\.?', 'www.', text)   # www 보정
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def simplify_domain_text(text):
    return re.sub(r'[^a-z0-9]', '', str(text).lower())

def find_best_official_domain(compact_text, official_domains_clean):
    best_match = ""
    best_window = ""
    min_distance = 999

    for official in official_domains_clean:
        official_compact = simplify_domain_text(official)
        if not official_compact:
            continue

        target_len = len(official_compact)

        # OCR 문자열 안에 공식 도메인이 그대로 포함된 경우
        if official_compact in compact_text:
            return official, official_compact, 0

        if len(compact_text) < target_len:
            distance = get_levenshtein_distance(compact_text, official_compact)
            if distance < min_distance:
                min_distance = distance
                best_match = official
                best_window = compact_text
            continue

        # 공식 도메인 길이 기준 +-2 범위로 윈도우 길이를 바꿔가며 비교
        # (OCR이 글자를 누락/중복 인식해서 길이가 살짝 달라지는 경우까지 포착)
        window_lengths = range(max(1, target_len - 2), target_len + 3)
        for window_len in window_lengths:
            if len(compact_text) < window_len:
                continue
            for start in range(0, len(compact_text) - window_len + 1):
                window = compact_text[start:start + window_len]
                distance = get_levenshtein_distance(window, official_compact)
                if distance < min_distance or (
                    distance == min_distance
                    and len(official_compact) > len(simplify_domain_text(best_match))
                ):
                    min_distance = distance
                    best_match = official
                    best_window = window

    return best_match, best_window, min_distance


def restore_dot_by_official(candidate_compact, official):
    """점이 빠진 문자열에, 매칭된 공식 도메인의 TLD 길이를 기준으로 점을 복원"""
    official_parts = str(official).split('.')
    if len(official_parts) < 2:
        return candidate_compact

    last_part_len = len(official_parts[-1])
    if len(candidate_compact) <= last_part_len:
        return candidate_compact

    return candidate_compact[:-last_part_len] + '.' + candidate_compact[-last_part_len:]


def find_best_domain_suffix(raw_domain, official_domains_clean):
    parts = raw_domain.split('.')
    suffixes = ['.'.join(parts[i:]) for i in range(0, len(parts) - 1)]

    best_suffix = raw_domain
    min_distance = 999

    for suffix in suffixes:
        suffix_compact = simplify_domain_text(suffix)

        for official in official_domains_clean:
            official_compact = simplify_domain_text(official)
            distance = get_levenshtein_distance(suffix_compact, official_compact)

            if distance < min_distance:
                min_distance = distance
                best_suffix = suffix

    return best_suffix, min_distance

def extract_domain(text):
    text_norm = normalize_url_text(text)

    # 공식 도메인이 OCR 텍스트 안에 그대로 들어있는 경우
    for official in official_domains_clean:
        official_pattern = r'(?<![a-z0-9.\-])' + re.escape(official) + r'(?![a-z0-9.\-])'
        if re.search(official_pattern, text_norm):
            return official

    domain_match = re.search(r'([a-z0-9\-]+(?:\.[a-z0-9\-]+)+)', text_norm)
    if domain_match:
        raw_domain = domain_match.group(1)
        #도메인 부분만 추출
        for official in official_domains_clean:
            if official in raw_domain:
                return official
        best_suffix, min_distance = find_best_domain_suffix(raw_domain, official_domains_clean)
        if best_suffix and min_distance <= 2:
            return best_suffix

        return raw_domain

    # OCR이 점(.)을 누락한 경우 공식 도메인 DB 기준으로 복구
    compact_text = simplify_domain_text(text_norm)
    if not compact_text:
        # 영문/숫자 후보가 전혀 없음 (순수 한글 문장, 빈 텍스트 등) -> 분석 대상 아님
        return None

    best_match, best_window, min_distance = find_best_official_domain(compact_text, official_domains_clean)
    if best_match and min_distance == 0:
        return best_match
    if best_match and best_window and min_distance <= 2:
        return restore_dot_by_official(best_window, best_match)

    # 화이트리스트 어떤 도메인과도 충분히 비슷하지 않음
    # -> 정제 안 된 뭉치 문자열을 도메인이라 우기지 않고 None으로 명확히 처리
    return None

# ---------------- 학습 데이터 도메인 생성 ----------------
df['domain'] = df['text'].astype(str).apply(extract_domain)
# extract_domain이 None을 반환하는 행은 학습에서 제외 (도메인 후보가 아니므로)
df = df.dropna(subset=['domain'])

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,4))
X = vectorizer.fit_transform(df['domain'].astype(str))
y = df['label']
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# ---------------- 판정 함수 ----------------
def classify_text(text):
    domain = extract_domain(text)

    # -1. 도메인 후보 자체가 없는 경우 (None) -> 가장 먼저 처리
    if domain is None:
        return ("⚠️ 도메인 인식 실패", text, "유효한 URL/도메인 패턴을 찾지 못함", 0.0)

    # 0. 레이블 CSV 직접 매칭
    match = df.loc[df['domain'] == domain]
    if not match.empty:
        label = match['label'].values[0]
        if label == 0:
            return ("✅ 정상", domain, "CSV 직접 매칭", 0.0)
        else:
            return ("🚨 피싱 위험", domain, "CSV 직접 매칭", 100.0)

    # 1. 화이트리스트 비교
    if domain in official_domains_db:
        return ("✅ 정상", domain, "화이트리스트 등록", 0.0)

    is_spoofing=False
    matched_official=""
    min_distance=999

    for official in official_domains_db:
        # 상단 셀에서 직접 구현한 get_levenshtein_distance 함수 호출
        domain_compact = simplify_domain_text(domain)
        official_compact = simplify_domain_text(official)
        distance = get_levenshtein_distance(domain_compact, official_compact)
        if 0< distance <=2:
            is_spoofing=True
            if distance < min_distance:
                min_distance = distance
                matched_official = official
    if is_spoofing:
        return ("🚨 피싱 위험", domain, f"편집거리 기반 사칭 도메인 탐지 (공식 '{matched_official}' 사칭 의심)", round(min_distance/len(matched_official)*100,1))

    # 2. CSV 학습 기반 판정
    X_test = vectorizer.transform([domain])
    pred = model.predict(X_test)[0]
    prob = model.predict_proba(X_test)[0][pred]
    if pred == 1:
        return ("🚨 피싱 위험", domain, "CSV 학습 기반 판정", round(prob*100,1))
    else:
        return ("⚠️ 미등록 도메인", domain, "CSV 학습 기반 판정", round(prob*100,1))

# ---------------- 반복 업로드 -> Gradio 인터페이스 연동 ----------------
reader = easyocr.Reader(['ko','en'], gpu=True)

# 그라디오용 안전 래핑 함수 (리턴값 매핑 보정)
def web_phishing_scan(image):
    if image is None:
        return "⚠️ 시연할 웹사이트/문자 캡처 이미지를 업로드해 주세요."

    ocr_result = reader.readtext(image, detail=0)
    extracted_text = " ".join(ocr_result)

    if not extracted_text.strip():
        return "🍏 안전: 이미지 내 추출된 텍스트 없음"

    # 원본 판정 함수 실행 후 결과를 4개 변수로 정확히 쪼개서 받음
    status, clean, detail, prob = classify_text(extracted_text)

    # 원본 print 양식 스타일로 최종 화면 출력
    return f"⚖️ {status}\n🧼 {clean}\n🎯 위험도 {prob}%\n📝 {detail}"

# 그라디오 UI 레이아웃 조립
demo = gr.Interface(
    fn=web_phishing_scan,
    inputs=gr.Image(type="filepath", label="📸 웹사이트 캡처 이미지 업로드"),
    outputs=gr.Textbox(label="🚨 실시간 분석 결과", lines=5),
    title="🛡️ 편집거리 기반 url 탐지 시연 모듈",
    description="스크린샷 이미지를 올리면 내부 EasyOCR 엔진이 주소를 스캔하여 정식 도메인 DB 및 CSV 기반 학습 모델을 거쳐 실시간 판정을 내립니다.",
    theme="soft"
)

# 외부 터널 링크 개방 및 실행
demo.launch(share=True, inline=True)

Mounted at /content/drive


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% CompleteColab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7672f7b817c177d6ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
